# Features for Churn Prediction

**Goal:** for every customer and snapshot date, build features using only data before the cutoff date.
**Input:** `sales.parquet`, `returns.parquet` and `labels.parquet`. **Output:** `features.parquet`.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

# Show up to 50 columns when printing a table
pd.set_option("display.max_columns", 50)

# One clean style for all charts
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)

# Folder where the charts are saved (they will be used in the README)
fig_dir = Path("../reports/figures")
fig_dir.mkdir(parents=True, exist_ok=True)


def takeaway(text):
    # Show the takeaway under a chart. The numbers are computed from the data.
    display(Markdown("**Takeaway:** " + text))

In [4]:
# Load the cleaned tables and the labels from the previous notebooks
sales = pd.read_parquet("../data/processed/sales.parquet")
returns = pd.read_parquet("../data/processed/returns.parquet")
labels = pd.read_parquet("../data/processed/labels.parquet")

# Sort by date once, so "last value" lookups later are correct
sales = sales.sort_values("invoicedate").reset_index(drop=True)

# Revenue of each sales row = quantity x price
sales["line_total"] = sales["quantity"] * sales["price"]

# Value of each return row (quantity is negative in returns, so take the absolute value)
returns["return_value"] = (returns["quantity"] * returns["price"]).abs()

print("Sales:", sales.shape, "| Returns:", returns.shape, "| Labels:", labels.shape)

Sales: (775278, 10) | Returns: (16285, 10) | Labels: (36297, 4)


## Build an invoice-level table

In [5]:
# One row per invoice (one purchase = one invoice)
invoices = (
    sales.groupby(["customer_id", "invoice"])
    .agg(
        invoice_date=("invoicedate", "min"),
        revenue=("line_total", "sum"),
        lines=("stockcode", "size"),    # number of product lines in the invoice
        items=("quantity", "sum"),      # number of items in the invoice
    )
    .reset_index()
)

# The calendar day of each invoice (several invoices on one day count as one purchase day)
invoices["invoice_day"] = invoices["invoice_date"].dt.normalize()

print("Invoices:", f"{len(invoices):,}", "| Customers:", f"{invoices['customer_id'].nunique():,}")
labels.head()

Invoices: 36,480 | Customers: 5,851


,customer_id,cutoff,split,churn
0,12346,2010-06-14,train,0
1,12349,2010-06-14,train,1
2,12355,2010-06-14,train,1
3,12358,2010-06-14,train,1
4,12359,2010-06-14,train,0
